# Field Shift — inventario de datos

Qué hay descargado, qué falta y qué conceptos se pueden evaluar ya. No entrena nada.

In [ ]:
import matplotlib.pyplot as plt
import polars as pl

from fieldshift_ml.datasets import registry
from fieldshift_ml.evaluation import plots
from fieldshift_ml.io import inventory, power

plots.style()
pl.Config.set_tbl_rows(40)
pl.Config.set_fmt_str_lengths(90)

## Fuentes en disco

In [ ]:
inventory.inventory()

## Conceptos: ¿se pueden construir?

Esta celda arma la tabla de cada concepto. La primera vez tarda (lee rasters y deja copias en caché).

In [ ]:
registry.status()

## Un vistazo: clima diario de NASA POWER

In [ ]:
daily = power.read_daily_all()
yearly = daily.group_by("site", pl.col("date").dt.year().alias("year")).agg(
    pl.col("PRECTOTCORR").sum().alias("lluvia_mm"), pl.col("T2M").mean().alias("temp_c"), pl.len().alias("dias"),
).filter(pl.col("dias") >= 360).sort("year")

fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
for site, color in zip(yearly["site"].unique().sort(), (plots.BLUE, plots.ORANGE), strict=False):
    part = yearly.filter(pl.col("site") == site)
    axes[0].plot(part["year"], part["lluvia_mm"], color=color, linewidth=1.8, label=site)
    axes[1].plot(part["year"], part["temp_c"], color=color, linewidth=1.8)
axes[0].set_title("Lluvia anual (mm)")
axes[1].set_title("Temperatura media anual (°C)")
axes[0].legend(ncols=2, loc="upper left")
for ax in axes:
    ax.grid(axis="x", visible=False)
fig.tight_layout()
plt.show()